# Redis exercises: from one server to Cluster

You can either follow this guide while reading [the brief Redis summary](https://github.com/tam-nguyen-3/redis-primer/blob/main/README.md) or follow along as you read; each section here matches one of its sections and lets you see the claim happen on your own machine.

**How to use this notebook**

- Run the cells in order. Each section starts with `reset()`, so you can also jump to any section.
- Cells marked **Checkpoint** requires your input! Predict the answer before you run anything, then check it. Answers are hidden under *Show answer*.
- You don't need to complete the Optional part

Run these two cells to install Redis and set up this notebook!

### Installing Redis

In [ ]:
# @title
%%bash
set -euo pipefail
curl -fsSL https://packages.redis.io/gpg | gpg --batch --yes --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
. /etc/os-release
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb ${VERSION_CODENAME} main" > /etc/apt/sources.list.d/redis.list
apt-get update -qq
apt-get install -y -qq redis
python -m pip install -q 'redis==8.1.0'
redis-server --version

In [ ]:
# @title Lab infrastructure — run this cell; no need to study it
import os
import pathlib
import re
import shutil
import signal
import socket
import subprocess
import threading
import time

import redis
from redis.backoff import NoBackoff
from redis.retry import Retry

assert shutil.which("redis-server"), "Run the Colab installation cell above first."
LAB = pathlib.Path("redis-lab").resolve()
# Preserve ownership if this setup cell is rerun in the same runtime.
_servers = globals().get("_servers", {})
_frozen = globals().get("_frozen", {})


def connect(port, **options):
    """Create a localhost client; surface failures without automatic retries."""
    defaults = dict(host="127.0.0.1", port=port, decode_responses=True,
                    socket_timeout=5, socket_connect_timeout=2,
                    retry=Retry(NoBackoff(), 0))
    defaults.update(options)
    return redis.Redis(**defaults)


def is_up(port):
    with socket.socket() as sock:
        sock.settimeout(0.2)
        return sock.connect_ex(("127.0.0.1", port)) == 0


def wait_until(condition, description, timeout=30):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            if condition():
                return
        except (redis.ConnectionError, redis.TimeoutError):
            pass
        time.sleep(0.1)
    raise TimeoutError(f"Timed out: {description}. See logs in {LAB}; rerun this section's setup.")


def launch(port, arguments):
    if port in _servers and _servers[port].poll() is None:
        raise RuntimeError(f"Lab server on {port} is already running; call reset_lab() first.")
    if is_up(port):
        raise RuntimeError(f"Port {port} is occupied by another process. It was not stopped. Restart the Colab runtime.")
    folder = LAB / str(port)
    folder.mkdir(parents=True, exist_ok=True)
    process = subprocess.Popen(
        ["redis-server", *arguments, "--port", str(port), "--bind", "127.0.0.1",
         "--dir", str(folder), "--logfile", str(folder / "log"), "--daemonize", "no"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    _servers[port] = process
    def ready():
        if process.poll() is not None:
            raise RuntimeError(f"Redis exited on port {port}; inspect {folder / 'log'}")
        return connect(port).ping()
    wait_until(ready, f"Redis on port {port} to start", timeout=15)


def start_server(port, *arguments):
    launch(port, ["--repl-diskless-sync-delay", "0", *arguments])


def start_sentinel(port, primary_port=6379, quorum=2):
    folder = LAB / str(port)
    folder.mkdir(parents=True, exist_ok=True)
    config = folder / "sentinel.conf"
    config.write_text(
        f"port {port}\n"
        f"sentinel monitor mymaster 127.0.0.1 {primary_port} {quorum}\n"
        "sentinel down-after-milliseconds mymaster 3000\n"
        "sentinel failover-timeout mymaster 10000\n")
    launch(port, [str(config), "--sentinel"])


def wait_synced(*ports):
    for port in ports:
        wait_until(lambda: connect(port).info("replication").get("master_link_status") == "up",
                   f"replica {port} to connect", timeout=45)


def start_replication_lab():
    reset_lab()
    start_server(6379, "--save", "", "--appendonly", "no")
    for port in (6380, 6381):
        start_server(port, "--save", "", "--appendonly", "no",
                     "--replicaof", "127.0.0.1", "6379")
    wait_synced(6380, 6381)


def sh(command):
    result = subprocess.run(command, shell=True, executable="/bin/bash",
                            capture_output=True, text=True, timeout=90)
    print(re.sub(r"\x1b\[[0-9;]*m", "", result.stdout + result.stderr).strip())
    result.check_returncode()


def owned_process(port):
    process = _servers.get(port)
    if process is None or process.poll() is not None:
        raise RuntimeError(f"No running notebook-owned server on {port}")
    return process


def freeze(port):
    process = owned_process(port)
    process.send_signal(signal.SIGSTOP)
    _frozen[port] = process
    print(f"Paused server on {port}")


def resume(port):
    process = _frozen.pop(port, None)
    if process is not None and process.poll() is None:
        process.send_signal(signal.SIGCONT)
        print(f"Resumed server on {port}")


def crash(port):
    process = owned_process(port)
    process.kill()
    process.wait(timeout=5)
    _frozen.pop(port, None)
    print(f"Crashed server on {port}")


def log(port, *words):
    for line in (LAB / str(port) / "log").read_text().splitlines():
        if not words or any(word.lower() in line.lower() for word in words):
            print(line)


def writer(write, n, delay=0.5):
    lines = []
    def run():
        for i in range(1, n + 1):
            try:
                status = f"OK {write(i) or ''}"
            except (redis.RedisError, OSError) as error:
                status = f"FAILED ({type(error).__name__}: {str(error).splitlines()[0]})"
            lines.append(f"{time.strftime('%H:%M:%S')}  write {i:3d}  {status}")
            time.sleep(delay)
    thread = threading.Thread(target=run, daemon=True)
    thread.start()
    return thread, lines


def reset_lab():
    """Stop only tracked child processes, then remove this lab's disposable files."""
    for port in list(_frozen):
        resume(port)
    for process in _servers.values():
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)
    _servers.clear()
    shutil.rmtree(LAB, ignore_errors=True)
    print("Lab cleared.")

print("Setup ready. Continue to section 1.")

## 1. What is Redis?

*Primer: What is Redis?* Redis is a key-value store that keeps everything in memory, and it runs commands one at a time.

In [ ]:
# Prepare a disposable server with persistence disabled.
reset_lab()
start_server(6379, "--save", "", "--appendonly", "no")
db = connect(6379)

A Python call such as `db.get(...)` sends the corresponding Redis command to the server.

In [ ]:
db.set("user:1:name", "Ada")
db.hset("user:1", mapping={"name": "Ada", "lang": "python"})
db.incr("page_views")

print("Stored name:", db.get("user:1:name"))
print("User profile:", db.hgetall("user:1"))
print("Page views:", db.get("page_views"))

### Checkpoint 1. TTL (Time-To-Live)

We can set the expiration to a key! Predict the last output then run the cell. If this were a cached profile, where would the application retrieve the profile after the key expires?

In [ ]:
expiration_seconds = 10  # Your edit: change this to 2.
db.set("cache:user:1:profile", "Ada's profile", ex=expiration_seconds)
print("Time remaining:", db.ttl("cache:user:1:profile"), "seconds")
time.sleep(3)
print("Cached profile after 3 seconds:", db.get("cache:user:1:profile"))

## 2. Transactions

Suppose we have 100 clients, each add 1 to `page_viewed`.

First the wrong way: read with `GET`, add 1 in Python, write back with `SET` (the `get_then_set` function, which is basically `x = x + 1`).

The other way is to use Redis' `INCR`, which is **atomic**: if multiple clients attempt `INCR` the same key concurrently, they will never cause a race condition.

The function `run_100_clients(increment)` will return the value of the keys, with `increment` being the different ways to increment a key's value described above.

In [ ]:
def run_100_clients(increment):
    db.set("page_viewed", 0)
    threads = [threading.Thread(target=increment) for _ in range(100)]
    for t in threads: t.start()
    for t in threads: t.join()
    return db.get("page_viewed")

def get_then_set():
    v = int(db.get("page_viewed"))
    time.sleep(0.001) # a tiny delay, like real application code between the read and the write
    db.set("page_viewed", v + 1)

def incr():
    db.incr("page_viewed")

### Checkpoint 2

Before running the code, can you predict the results of using `get_then_set` and using Redis' `INCR` to increment the value of a key?

In [ ]:
print("GET + SET:", run_100_clients(get_then_set))
print("INCR:     ", run_100_clients(incr))

### Optional: using `MULTI/EXEC` blocks, no rollback (unlike SQL), and `WATCH` for locking a key

#### **MULTI / EXEC: no rollback.**
Commands are queued and then run back to back, but if one fails at runtime, the others still run.


In [ ]:
# Prepare a disposable server with persistence disabled.
reset_lab()
start_server(16379, "--save", "", "--appendonly", "no")
db = connect(16379)

In this example, we're attempting to increment the value of `name` and `page_views`.

In [ ]:
db.set("name", "ada")
db.set("page_views", 0)

# transaction=True wraps the queued commands in MULTI ... EXEC.
with db.pipeline(transaction=True) as transaction:
    transaction.incr("name")        # Runtime error: "ada" is not a number.
    transaction.incr("page_views")  # Still executes.
    results = transaction.execute(raise_on_error=False)

print("First command:", results[0])
print("Second command:", results[1])
print("Final page_views:", db.get("page_views"))

#### **WATCH: optimistic locking.**

`WATCH key` makes the next `EXEC` fail if anyone changed the key in between. Here A and B race for the last ticket, and B sneaks in between A's read and A's write.

In [ ]:
reset_lab()
start_server(16379, "--save", "", "--appendonly", "no")

# set up two clients, which connects to the same Redis server
client_a = connect(16379)
client_b = connect(16379)
client_a.set("tickets", 1)

In [ ]:
with client_a.pipeline() as transaction:
    # Watch for changes before A commits its transaction.
    transaction.watch("tickets")
    print("A sees", transaction.get("tickets"), "ticket(s)")

    # B buys the last ticket before A commits.
    client_b.decr("tickets")
    print("B buys it first -> tickets =", client_b.get("tickets"))

    # Queue A's purchase, then attempt to commit.
    transaction.multi()
    transaction.decr("tickets")

    try:
        transaction.execute()
        print("A: EXEC succeeded")
    except redis.WatchError:
        print("A: EXEC aborted because tickets changed after WATCH")

print("Final tickets =", client_a.get("tickets"))

## 3. Persistence

*Primer: Persistence.* If there is a partition (e.g., node crashing, network error), how does Redis persist data?

**RDB**: a snapshot at one moment. Writes after the snapshot are lost.

In [ ]:
reset_lab()

# --save: disable automatic RDB snapshotting
rdb_options = ["--save", "", "--appendonly", "no"]
start_server(6379, *rdb_options)
db = connect(6379)
db.set("a", 1)
db.set("b", 2)

# we ask Redis to create a RDB snapshot after setting "a" and "b".
db.save()

# then we write "c".
db.set("c", 3)

### Checkpoint 3. RDB Persistence
We write the key `c` after creating the RDB snapshot. If the server crashes, what happens to the write? Predict what happens, then run the code cell below.

In [ ]:
crash(6379)
start_server(6379, *rdb_options)
print("Keys recovered from the snapshot:", sorted(connect(6379).keys("*")))

<details><summary>Show explanation </summary>

`db.set("c", 3)`: Writes a third key, c. Since this occurs after the manual snapshot, c resides only in memory and would be lost if the server crashed immediately without another save.

Thus, only `a` and `b`, which was in the snapshot, returns.

</details>

### Optional: AOF (Append-only File)

**AOF**: a log of every write command, like a write-ahead log.



**Your turn 3.1.** The primer says AOF with `everysec` "can lose ~1 second". Predict: if we crash *immediately* after the 100 `INCR`s, with no `sleep(1)`, what will `counter` be after restart? Then run the cell.

In [ ]:
reset_lab()
aof_options = ["--save", "", "--appendonly", "yes", "--appendfsync", "everysec"]
start_server(6379, *aof_options)
db = connect(6379)
for _ in range(100):
    db.incr("counter")
crash(6379)
start_server(6379, *aof_options)
print("Counter after process restart:", connect(6379).get("counter"))

<details><summary>Show answer</summary>

Still 100. Before replying OK, Redis has already *written* each command to the file. The writes sit in the operating system's buffer, waiting for the once-a-second `fsync` to push them to the physical disk. `kill -9` kills only the Redis process, and the operating system still finishes writing its buffer.

The ~1 second of loss happens when the **whole machine** goes down (power cut, kernel crash) before `fsync`. So "acknowledged ≠ durable" is about the machine dying, not the process. `--appendfsync always` closes that gap, at the cost of an `fsync` per write.
</details>

## 4. Replication

*Primer: Replication.* One primary (6379) and two replicas (6380, 6381). Each server reports a **replication ID** and an **offset**.

In [ ]:
# creates 1 leader (at port 6379) and two followers (6380, 6381)
start_replication_lab()

In [ ]:
# Keep one connection so SET and WAIT refer to the same connection's writes.
primary = connect(6379, single_connection_client=True)
primary.set("greeting", "hello")
print("Replica acknowledgments:", primary.wait(2, 2000))
for port in (6379, 6380, 6381):
    node = connect(port)
    print(f"Port {port}: role={node.info('replication')['role']}, greeting={node.get('greeting')}")

### Checkpoint 4.

We pause both replicas and write to the primary (`freeze`).

Note: `WAIT` (or `.wait()` here) asks Redis: "How many replicas have acknowledged my earlier writes?"

If `SET` succeeds but `WAIT 2` returns 0, was the write undone?

What could happen if the primary failed before either replica received it? **Answer in one or two sentences.**

In [ ]:
freeze(6380)
freeze(6381)
try:
    accepted = primary.set("order:2", "paid")
    acknowledged = primary.wait(2, 1000)
    print("SET succeeded:", accepted)
    print("Replicas that acknowledged the write:", acknowledged)
    print("Value still on primary:", primary.get("order:2"))
finally:
    resume(6380)
    resume(6381)

print("Acknowledgments after replicas resume:", primary.wait(2, 3000))

<details><summary>Show explanation — open after predicting</summary>

The write is not undone: `WAIT` reports acknowledgments, not rollback. If the primary failed before either replica received the write, a promoted replica could lack the acknowledged order update. Even a successful `WAIT` does not turn Redis replication into a consensus protocol or guarantee that a write survives every failover. [Redis WAIT documentation](https://redis.io/docs/latest/commands/wait/)

</details>

### Optional: Replication internals -- partial and full resynchronization with master node

In [ ]:
start_replication_lab()
primary = connect(6379, single_connection_client=True)
primary.set("greeting", "hello")
primary.wait(2, 2000)
for port in (6379, 6380, 6381):
    info = connect(port).info("replication")
    print(port, info["role"], "ID", info["master_replid"][:12], "offset", info["master_repl_offset"])
log(6380, "resync", "sync:")

**Partial resync.** Pause a replica and disconnect the replication links. After reconnection, inspect the logs for a partial resync: the primary can send the missing bytes while they still fit in its backlog. The link-kill command below disconnects both replicas; only 6380 stays paused.

In [ ]:
freeze(6380)
primary.client_kill_filter(_type="replica")      # drop the replication links
for i in range(100): primary.set(f"k:{i}", "v")  # 6380 misses these
resume(6380)
wait_synced(6380, 6381)
primary.wait(2, 3000)
log(6380, "partial")
log(6379, "accepted")
print("replica keys:", connect(6380).dbsize(), " primary keys:", primary.dbsize())

**Backlog experiment.** Request a 16 KB backlog, then write 60 KB while replica 6380 is away. Predict whether it can catch up with a partial resync. Inspect the newest log lines after running; allocation details can make the actual retained history larger than the requested size.

In [ ]:
time.sleep(2)
primary.config_set("repl-backlog-size", "16kb")
freeze(6380)
primary.client_kill_filter(_type="replica")
for i in range(3): primary.set(f"big:{i}", "x" * 20_000)
resume(6380)
time.sleep(3)
log(6379, "6380")

**Your turn 4.1.** The backlog holds 1 MB by default. Shrink it to 16 KB and write 60 KB while 6380 is away. Predict: partial or full resync? Run the cell and read the primary's log.

<details><summary>Show answer</summary>

Full. The primary logs `Unable to partial resync ... for lack of backlog`: the 60 KB of missed writes no longer fit in the 16 KB backlog, so the offset the replica asked for has already been overwritten. The primary sends a whole RDB snapshot instead. Backlog size decides how long a replica can be disconnected before it needs a full copy.
</details>

## 5. Redis Cluster - Cluster hash tags

*Primer: Redis Cluster.* Six nodes: three primaries, each with one replica. 16,384 slots are split across the primaries.

In [ ]:
from redis.cluster import key_slot

def slot_for(key):
    return key_slot(key.encode("utf-8"))

for key in ("user:123:profile", "user:123:posts"):
    print(f"{key} - slot {slot_for(key)}")

### Checkpoint 6.

Rename the two keys below so they share a slot, using a hash tag for this user's ID. Think briefly: why could using the same tag for *every* user overload part of the cluster? (1-2 sentences)

In [ ]:
profile_key = "user:123:profile"  # Your edit: add a user-specific hash tag.
posts_key = "user:123:posts"      # Your edit: use the same tag here.

print("Profile slot:", slot_for(profile_key))
print("Posts slot:", slot_for(posts_key))
if slot_for(profile_key) == slot_for(posts_key):
    print("Same slot: these keys can be used together in a native Cluster MGET.")
else:
    print("Different slots: edit the names and try again.")

<details><summary>Show explanation — open after predicting</summary>

Use `{user:123}:profile` and `{user:123}:posts`. Redis hashes the shared text inside the braces. A different user can have a different tag; assigning every user the same tag concentrates all those keys in one slot, creating a potential hot spot.

</details>

### Optional - Live Redis Cluster

In [ ]:
reset_lab()
CLUSTER = ["--save", "", "--cluster-enabled", "yes", "--cluster-config-file", "nodes.conf",
           "--cluster-node-timeout", "3000", "--appendonly", "yes"]
for p in range(7000, 7006):
    start_server(p, *CLUSTER)
sh("redis-cli --cluster create " + " ".join(f"127.0.0.1:{p}" for p in range(7000, 7006))
   + " --cluster-replicas 1 --cluster-yes | grep -E 'Slots|Adding replica|All 16384'")
wait_until(lambda: connect(7000).cluster("info")["cluster_state"] == "ok",
           "Cluster to become healthy", timeout=45)
wait_synced(*[p for p in range(7000, 7006) if connect(p).role()[0] == "slave"])

In [ ]:
from redis.cluster import RedisCluster
cluster_client = RedisCluster(host="127.0.0.1", port=7000, decode_responses=True)

try:
    connect(7000).set("foo", "bar")          # plain client, talking to 7000 only
except redis.ResponseError as e:
    print("plain client:", e)

for key in ["foo", "hello", "user:123:profile", "user:123:posts"]:
    cluster_client.set(key, "x")
    node = cluster_client.get_node_from_key(key)
    print(f"{key:18s} slot {cluster_client.keyslot(key):5d} -> {node.port}")

In [ ]:
for i in range(3000):
    cluster_client.set(f"key:{i}", i)
sh("redis-cli --cluster check 127.0.0.1:7000 | grep 'keys |'")

**Hash tags in a live cluster.** Compare `MGET` (multiple get -- getting multiple keys) with and without hash tags.

In [ ]:
try:
    cluster_client.mget("user:123:profile", "user:123:posts")
except redis.RedisClusterException as e:
    print("without tags:", e)

In [ ]:
profile_key = "{user:123}:profile"
posts_key = "{user:123}:posts"
cluster_client.set(profile_key, "Ada")
cluster_client.set(posts_key, "3 posts")
print("Profile and posts:", cluster_client.mget(profile_key, posts_key))

**When a primary fails.** No Sentinel here: the other primaries vote, and a replica takes over the failed primary's slots.

In [ ]:
failed_port = cluster_client.get_node_from_key("foo").port
observer_port = next(port for port in range(7000, 7006) if port != failed_port)
failed_id = connect(failed_port).execute_command("CLUSTER", "MYID")

def slot_owner_changed():
    # Observe promotion directly: a former replica of this primary now owns slots.
    nodes = connect(observer_port).execute_command("CLUSTER", "NODES").splitlines()
    return any(line.split()[0] in replica_ids and "master" in line.split()[2]
               and len(line.split()) > 8 for line in nodes)

nodes_before = connect(observer_port).execute_command("CLUSTER", "NODES").splitlines()
replica_ids = {line.split()[0] for line in nodes_before if line.split()[3] == failed_id}
started = time.monotonic()
freeze(failed_port)
try:
    wait_until(slot_owner_changed, "a replica to take over the failed primary's slots", timeout=60)
    wait_until(lambda: connect(observer_port).cluster("info")["cluster_state"] == "ok",
               "Cluster to recover", timeout=30)
    print(f"Replica promotion observed after {time.monotonic() - started:.1f} seconds")
    sh(f"redis-cli -p {observer_port} CLUSTER NODES")
finally:
    resume(failed_port)
wait_until(lambda: connect(failed_port).role()[0] == "slave",
           "old Cluster primary to become a replica", timeout=60)
wait_synced(failed_port)

**Adding a node + Resharding** A new primary (7006) takes 1,000 slots. Only the keys in those slots move.

With roughly 3,000 uniformly distributed keys, approximately how many should move to it? Does adding a node require moving every key?

In [ ]:
start_server(7006, *CLUSTER)
sh("redis-cli --cluster add-node 127.0.0.1:7006 127.0.0.1:7000 | tail -1")
wait_until(lambda: all(int(connect(port).cluster("info")["cluster_known_nodes"]) == 7
                       for port in range(7000, 7007)),
           "all nodes to discover the new node", timeout=45)
new_id = connect(7006).execute_command("CLUSTER", "MYID")
sh(f"redis-cli --cluster reshard 127.0.0.1:7000 --cluster-from all --cluster-to {new_id} "
   "--cluster-slots 1000 --cluster-yes > /tmp/redis-lab-reshard.log 2>&1")
sh("redis-cli --cluster check 127.0.0.1:7000 | grep -E 'keys [|]|All 16384'")
print("Keys on new primary:", connect(7006).dbsize())

<details><summary>Show explanation — open after predicting</summary>

<details><summary>Show answer</summary>

Server 7006 owns 1,000 of 16,384 slots, about 6%. 6% of the ~3,000 keys (plus the few others) is about 180–190. The rest stayed where they were. With `hash(key) mod number_of_nodes`, going from 3 to 4 nodes would have moved about three quarters of all keys.
</details>

</details>

## 6. Optional: Sentinel

*Primer: Sentinel.* Three servers and three sentinels with quorum 2. A client keeps writing through Sentinel while we freeze the primary.

In [ ]:
start_replication_lab()
for port in (26379, 26380, 26381):
    start_sentinel(port)
wait_until(
    lambda: all(connect(port).sentinel_master("mymaster").get("num-other-sentinels", 0) == 2
                and connect(port).sentinel_master("mymaster").get("num-slaves", 0) == 2
                for port in (26379, 26380, 26381)),
    "Sentinels to discover each other and the replicas", timeout=45)

def current_primary():
    return int(connect(26379).sentinel_get_master_addr_by_name("mymaster")[1])

print("primary is", current_primary())
print({k: v for k, v in connect(26379).sentinel_master("mymaster").items()
       if k in ("num-slaves", "num-other-sentinels", "quorum")})

In [ ]:
from redis.sentinel import Sentinel
sentinel = Sentinel([("127.0.0.1", p) for p in (26379, 26380, 26381)], socket_timeout=0.5)
client = sentinel.master_for("mymaster", socket_timeout=0.5)

def write(i):
    client.set(f"tick:{i}", i)
    return "-> :%s" % sentinel.discover_master("mymaster")[1]

t, lines = writer(write, 30)        # 30 writes, one every 0.5 s
time.sleep(2)
freeze(6379)                        # the primary hangs
t.join()
print("\n".join(lines))
print("Primary after experiment:", current_primary())
print("Failed requests:", sum("FAILED" in line for line in lines))

**Read the election evidence.** Look for suspicion (`sdown`), agreement (`odown`), leader election, and a primary switch. Exact timing and the selected replica may differ between runs.

In [ ]:
for p in (26379, 26380, 26381):
    print(f"--- sentinel {p}")
    log(p, "sdown master", "odown", "elected-leader", "selected-slave", "switch-master")

The sentinel logs show each step: a sentinel suspects the primary (`+sdown`), the quorum agrees (`+odown`), they vote a leader (`+elected-leader`), and the leader picks a replica to promote (`+selected-slave`).

In [ ]:
resume(6379)
wait_until(
    lambda: connect(6379).role()[0] == "slave"
            and connect(6379).info("replication")["master_link_status"] == "up",
    "old primary to rejoin as a replica", timeout=60)
print("6379 is now a replica of", current_primary())

**Simulated split-brain.** Now the old primary isn't dead but cut off. We technically can't "cut network", so we fake a partition with users: the client logs in as `app`, while sentinels and replicas use the `default` user. Turning `default` off cuts them off while the client keeps working.

`split_brain()` counts the writes the client was told succeeded, then the writes that actually survived. It takes about 35 seconds.


Predict: can a client receive successful replies for writes that later disappear? Compare **acknowledged**, **survived**, and **lost** counts. Allow about 40 seconds per run.

In [ ]:
# @title Split-brain simulation helper — run without editing
def split_brain(seconds=15):
    P = current_primary()
    for p in (6379, 6380, 6381):
        connect(p).execute_command("ACL", "SETUSER", "app", "on", ">apppass", "~*", "&*", "+@all")
    app = redis.Redis(port=P, username="app", password="apppass", socket_timeout=0.5, decode_responses=True, single_connection_client=True, retry=Retry(NoBackoff(), 0))
    app.delete("log")
    wait_until(lambda: app.wait(2, 1000) == 2, "both replicas to acknowledge the initial state", timeout=45)
    acked = []
    def write(i):
        app.rpush("log", i)
        acked.append(i)
        return f"-> :{P} ({app.role()[0]})"

    t, lines = writer(write, 4 * seconds + 6)
    time.sleep(2)
    try:
        app.execute_command("ACL", "SETUSER", "default", "off")
        app.execute_command("CLIENT", "KILL", "USER", "default")
        print(f"Simulated partition starts at {time.strftime('%H:%M:%S')}")
        time.sleep(seconds)
    finally:
        app.execute_command("ACL", "SETUSER", "default", "on")
        print(f"Connectivity restored at {time.strftime('%H:%M:%S')}")
    t.join()

    wait_until(lambda: current_primary() != P and connect(P).role()[0] == "slave"
               and connect(P).info("replication")["master_link_status"] == "up",
               "partition healing and old-primary demotion", timeout=60)
    kept = connect(current_primary()).lrange("log", 0, -1)
    print("\n".join(lines[::4]))
    print(f"\nclient was told OK: {len(acked)} writes")
    print(f"survived:           {len(kept)} writes  {kept}")
    print(f"lost:               {len(set(map(str, acked)) - set(kept))} acknowledged writes")

In [ ]:
split_brain()

**Your turn 5.1.** Turn on `min-replicas-to-write 1` (with `min-replicas-max-lag 5`) on every server: a primary now refuses writes unless at least one replica acknowledged it within the last 5 seconds. Predict how many acknowledged writes will be lost, then run it.

In [ ]:
for p in (6379, 6380, 6381):
    connect(p).config_set("min-replicas-to-write", 1)
    connect(p).config_set("min-replicas-max-lag", 5)
split_brain()

<details><summary>Show answer</summary>

None, in this demo. Once the old primary loses its replicas, it answers `NOREPLICAS` instead of OK, so the client gets an error instead of a false success. The writes during the partition fail loudly instead of disappearing.

In our demo the replica links drop instantly. In a real partition, the old primary keeps accepting writes for up to `min-replicas-max-lag` seconds before it notices. So this setting shrinks the window for lost writes; it doesn't close it.
</details>

**Think about it (from the primer):** Sentinel *votes* on who becomes primary. Why doesn't Redis vote on each write too, and what would that cost?

## Clean up

In [ ]:
reset_lab()